# novaduck tour

One notebook reaches every catalog your role sees: Nova's own tables, Postgres and Snowflake. This tour reads orders from all three, turns them into revenue per place, and upserts the result into a Nova table and a Postgres table.

Everything runs as you, under your active role. Ranger decides what you may read and change, and every write is audited.

In [ ]:
import novaduck
import pandas as pd

con = novaduck.connect()
# Each catalog is attached by its Nova name: nova (Nova's own tables), pg_shop, sf_pat, ...
pd.DataFrame(con.catalogs())[["alias", "type", "attached", "error"]]

In [ ]:
SELECT 'nova' AS source,
       shipping_city AS place,
       CAST(order_id AS VARCHAR) AS order_id,
       CAST(total_amount AS DOUBLE) AS amount
FROM nova.NOVA_DEMO.orders
WHERE status <> 'pending'

In [ ]:
SELECT 'postgres' AS source,
       city AS place,
       CAST(id AS VARCHAR) AS order_id,
       CAST(amount AS DOUBLE) AS amount
FROM pg_shop.public.orders

In [ ]:
SELECT 'snowflake' AS source,
       REGION AS place,
       CAST(ORDER_ID AS VARCHAR) AS order_id,
       CAST(AMOUNT AS DOUBLE) AS amount
FROM sf_pat.SALES.ORDERS
WHERE AMOUNT < 1000000000

In [ ]:
SELECT * FROM {{nova_orders}}
UNION ALL
SELECT * FROM {{pg_orders}}
UNION ALL
SELECT * FROM {{sf_orders}}

In [ ]:
SELECT source,
       place,
       CAST(count(*) AS INTEGER) AS orders,
       CAST(round(sum(amount), 2) AS DECIMAL(18, 2)) AS revenue,
       CAST(round(sum(amount) / sum(sum(amount)) OVER (PARTITION BY source), 4) AS DECIMAL(7, 4)) AS share,
       CAST(now() AS TIMESTAMP) AS refreshed_at
FROM {{all_orders}}
GROUP BY source, place
ORDER BY source, revenue DESC

In [ ]:
df = place_revenue.df()
df.sort_values("revenue", ascending=False).groupby("source").head(1)

In [ ]:
# Nova DDL runs in Nova itself, as you. IF NOT EXISTS makes the cell safe to run again.
con.execute("""CALL mysql_execute('nova', '
    CREATE TABLE IF NOT EXISTS NOVA_DEMO.place_revenue (
        source VARCHAR(32) NOT NULL,
        place VARCHAR(128) NOT NULL,
        orders INT,
        revenue DECIMAL(18, 2),
        share DECIMAL(7, 4),
        refreshed_at DATETIME
    ) PRIMARY KEY (source, place)
    DISTRIBUTED BY HASH(source, place)')""")
"NOVA_DEMO.place_revenue is ready"

In [ ]:
# Rows with the same (source, place) are updated, new ones inserted.
con.merge_into("nova.NOVA_DEMO.place_revenue", place_revenue, on=["source", "place"])

In [ ]:
# Postgres and Snowflake are written through their catalog's write connection,
# which an administrator adds in Catalog. A plain SQL INSERT into them is refused.
con.merge_into("pg_shop.public.place_revenue", place_revenue, on=["source", "place"])

### Snowflake works the same way

When the Snowflake catalog has a write connection, the same call upserts there:

```python
con.merge_into("sf_pat.SALES.PLACE_REVENUE", place_revenue, on=["SOURCE", "PLACE"])
```

`sf_pat` reads with a read-only Snowflake role here, so this tour does not write to it.

In [ ]:
SELECT n.source,
       n.place,
       n.revenue AS nova_revenue,
       p.revenue AS postgres_revenue,
       n.revenue = p.revenue AS same
FROM nova.NOVA_DEMO.place_revenue AS n
FULL JOIN pg_shop.public.place_revenue AS p USING (source, place)
ORDER BY n.source, n.revenue DESC

### More you can do with `con`

| Call | What it does |
|---|---|
| `con.write(data, "nova.db.table")` | Append rows; `mode="overwrite"` replaces them |
| `con.merge_into(table, data, on=[...], when_matched="delete")` | Upsert, or delete matches (`"ignore"` leaves them) |
| `con.put(data, "@stage/folder/name.parquet")` | Save data or a file into a Nova stage (`.csv`, `.json` too) |
| `con.nova("SELECT ... FROM @stage/folder/name.parquet")` | Run Nova SQL in Nova, for Nova-only syntax |
| `con.catalogs()` | The catalogs attached here, and any that could not be reached |

`data` can be a SQL string, a DuckDB relation such as a SQL cell's result, a pandas, Polars or Arrow frame, or a local file path.